## GPU Accelerated 2D Heat Diffusion Solver using Numba CUDA

### GPU Programming Assignment

Author:
- Member 1
- Member 2
- Member 3
- Member 4

Course: GPU Programming

Problem 5:
Stencil-Based PDE Solver using GPU

## Problem Statement
The objective of this assignment is to implement a Partial Differential Equation (PDE) solver on a GPU.

We selected the 2D Heat Diffusion Equation because it is a classical PDE used to model heat transfer in physical systems.

The implementation uses:

- Python
- NumPy
- Numba CUDA

The performance of CPU and GPU implementations is compared.

## What is Heat Diffusion?

#### Heat Diffusion

Heat naturally flows from hotter regions to colder regions.

Examples:

- CPU cooling
- Battery thermal management
- Weather simulations
- Metal plate heating

The heat diffusion equation models this process mathematically.

## PDE

The heat quation is:
$$u(x,y,t): \text{temperature at position } (x,y) \text{ and time } t$$

$$\frac{\partial u}{\partial t} = \alpha \nabla^2 u = \alpha \left( \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} \right)$$

- **α (thermal diffusivity):** How fast heat spreads (material-dependent, e.g., α=0.1 for steel)
- **∇²u (Laplacian):** Measures how "curved" the temperature field is
  - If a point is hotter than neighbors → Laplacian is positive → temperature decreases
  - If a point is cooler than neighbors → Laplacian is negative → temperature increases

## Stencil Computation

### 5-Point Stencil

Each cell updates using four neighboring cells.

```
         Up

Left   Center   Right

         Down
```




In [ ]:
%pip install numpy numba matplotlib pandas

## Section 6 - CPU Implementation

In [4]:
import numpy as np

def cpu_heat_step(old_grid):

    rows, cols = old_grid.shape

    new_grid = old_grid.copy()

    for i in range(1, rows-1):
        for j in range(1, cols-1):

            new_grid[i,j] = (
                old_grid[i-1,j]
                + old_grid[i+1,j]
                + old_grid[i,j-1]
                + old_grid[i,j+1]
            ) * 0.25

    return new_grid 

## Section 7 - CPU Benchmark

In [9]:
import time

def run_cpu_heat_diffusion(
        grid_size=512,
        iterations=500,
        initial_temperature=100.0):

    # ------------------------------------------
    # Grid Initialization
    # ------------------------------------------

    grid = np.zeros(
        (grid_size, grid_size),
        dtype=np.float32
    )

    center = grid_size // 2

    grid[center, center] = initial_temperature

    # ------------------------------------------
    # Timing
    # ------------------------------------------

    start_time = time.time()

    for _ in range(iterations):
        grid = cpu_heat_step(grid)

    elapsed_time = time.time() - start_time

    return {
        "grid_size": grid_size,
        "iterations": iterations,
        "execution_time": elapsed_time,
        "result": grid
    }

## Section 8 - GPU Karnel

In [6]:
import numpy as np
import time
from numba import cuda

# =====================================================
# GPU KERNEL
# =====================================================
@cuda.jit
def heat_kernel(old_grid, new_grid):

    row, col = cuda.grid(2)

    rows = old_grid.shape[0]
    cols = old_grid.shape[1]

    # Skip boundary cells
    if 0 < row < rows - 1 and 0 < col < cols - 1:

        new_grid[row, col] = (
            old_grid[row - 1, col] +
            old_grid[row + 1, col] +
            old_grid[row, col - 1] +
            old_grid[row, col + 1]
        ) * 0.25

## Section 9 - GPU Benchmark

This section evaluates the performance of the CUDA-based heat diffusion solver.

The benchmark consists of:

1. Device memory allocation
2. Kernel warmup execution
3. Timed execution of multiple iterations
4. Collection of execution statistics

The first kernel launch is excluded from timing because it includes JIT compilation overhead.

In [ ]:
# =====================================================
# GPU CONFIGURATION
# =====================================================

THREADS_PER_BLOCK = (16, 16)

print("Threads Per Block:", THREADS_PER_BLOCK)

# =====================================================
# GPU BENCHMARK FUNCTION
# =====================================================

import time
from numba import cuda
import numpy as np


def run_gpu_heat_diffusion(
        grid_size=512,
        iterations=500,
        initial_temperature=100.0):

    # ------------------------------------------
    # Initialize Grid
    # ------------------------------------------

    grid = np.zeros(
        (grid_size, grid_size),
        dtype=np.float32
    )

    center = grid_size // 2

    grid[center, center] = initial_temperature

    # ------------------------------------------
    # Copy to Device
    # ------------------------------------------

    d_old = cuda.to_device(grid)
    d_new = cuda.device_array_like(grid)

    # ------------------------------------------
    # Grid Configuration
    # ------------------------------------------

    blocks_per_grid = (

        (grid_size + THREADS_PER_BLOCK[0] - 1)
        // THREADS_PER_BLOCK[0],

        (grid_size + THREADS_PER_BLOCK[1] - 1)
        // THREADS_PER_BLOCK[1]
    )

    # ------------------------------------------
    # Warmup Launch
    # ------------------------------------------

    heat_kernel[
        blocks_per_grid,
        THREADS_PER_BLOCK
    ](
        d_old,
        d_new
    )

    cuda.synchronize()

    # ------------------------------------------
    # Timed Execution
    # ------------------------------------------

    start_time = time.time()

    for _ in range(iterations):

        heat_kernel[
            blocks_per_grid,
            THREADS_PER_BLOCK
        ](
            d_old,
            d_new
        )

        d_old, d_new = d_new, d_old

    cuda.synchronize()

    elapsed_time = time.time() - start_time

    # ------------------------------------------
    # Copy Result Back
    # ------------------------------------------

    result = d_old.copy_to_host()

    return {
        "grid_size": grid_size,
        "iterations": iterations,
        "execution_time": elapsed_time,
        "result": result
    }

# =====================================================
# SINGLE GPU EXPERIMENT
# =====================================================

gpu_result = run_gpu_heat_diffusion(
    grid_size=512,
    iterations=500
)

print("Grid Size     :", gpu_result["grid_size"])
print("Iterations    :", gpu_result["iterations"])
print("GPU Time (s)  :", gpu_result["execution_time"])

# =====================================================
# INSPECT CENTER REGION
# =====================================================

result = gpu_result["result"]

N = result.shape[0]
center = N // 2

print(
    result[
        center-2:center+3,
        center-2:center+3
    ]
)

## Section 10 - Visualization

In [ ]:
# =====================================================
# VISUALIZATION
# =====================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))

plt.imshow(
    gpu_result["result"],
    cmap="hot"
)

plt.colorbar()

plt.title(
    f"Heat Diffusion ({gpu_result['grid_size']}x{gpu_result['grid_size']})"
)

plt.show()

## Usage Example

In [ ]:
cpu_result = run_cpu_heat_diffusion(
    grid_size=512,
    iterations=500
)

gpu_result = run_gpu_heat_diffusion(
    grid_size=512,
    iterations=500
)

print("CPU Time:", cpu_result["execution_time"])
print("GPU Time:", gpu_result["execution_time"])

## Section 11 — Speedup

Speedup formula:

Speedup = GPU Time / CPU Time
	​


In [ ]:
speedup = (
    cpu_result["execution_time"] / gpu_result["execution_time"]
)

print(f"Speedup: {speedup:.2f}x")

## Section 12 -  Experiment

In [ ]:
import pandas as pd

sizes = [256, 512, 1024]

results = []

for size in sizes:

    cpu_result = run_cpu_heat_diffusion(
        grid_size=size,
        iterations=500
    )

    gpu_result = run_gpu_heat_diffusion(
        grid_size=size,
        iterations=500
    )

    speedup = (
        cpu_result["execution_time"]
        /
        gpu_result["execution_time"]
    )

    results.append({
        "Grid Size": f"{size}x{size}",
        "CPU Time (s)": round(
            cpu_result["execution_time"], 4
        ),
        "GPU Time (s)": round(
            gpu_result["execution_time"], 4
        ),
        "Speedup": round(speedup, 2)
    })

df = pd.DataFrame(results)

df

## Section 13 - Conclusion


A stencil-based PDE solver for the 2D heat diffusion equation was implemented using both CPU and GPU approaches.

The GPU implementation used CUDA through the Numba framework. Experimental results showed that the GPU significantly reduced execution time compared to the CPU implementation.

This demonstrates the suitability of GPUs for stencil computations and PDE simulations because many grid cells can be updated simultaneously.